In [1]:
# ----------------------------
# Cell 1 — Imports and paths
# ----------------------------
import gc
import glob
import numpy as np
import pandas as pd
import gseapy as gp
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
DE_RESULTS_DIR = PROJECT_DIR / "results" / "phase4_finelabels_DE"
RESULTS_DIR = PROJECT_DIR / "results" / "phase4_finelabels_pathways"
FIGURE_DIR = PROJECT_DIR / "figures" / "phase4_finelabels_pathways"
for d in [RESULTS_DIR, FIGURE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Setup complete")

Setup complete


In [2]:
# ----------------------------
# Cell 2 — Auto-discover all DE result files (full, not just
# significant), run enrichment per comparison, background scoped to
# genes actually tested in that specific comparison (same principle as
# original notebook 06).
# ----------------------------
de_files = sorted(glob.glob(str(DE_RESULTS_DIR / "*_tumor_vs_normal.csv"))) + \
           sorted(glob.glob(str(DE_RESULTS_DIR / "*_TNBC_vs_ER+.csv"))) + \
           sorted(glob.glob(str(DE_RESULTS_DIR / "*_HER2+_vs_ER+.csv"))) + \
           sorted(glob.glob(str(DE_RESULTS_DIR / "*_TNBC_vs_HER2+.csv")))
de_files = [f for f in de_files if "_significant" not in f]

print(f"Found {len(de_files)} DE result files to process")
for f in de_files[:5]:
    print(f"  {Path(f).name}")

Found 60 DE result files to process
  GSE114725_finelabels_DE_Activated_CD8_T_cells_tumor_vs_normal.csv
  GSE114725_finelabels_DE_B_cells_tumor_vs_normal.csv
  GSE114725_finelabels_DE_CD4_Activated_T_cells_tumor_vs_normal.csv
  GSE114725_finelabels_DE_Monocyte-like_macrophages_tumor_vs_normal.csv
  GSE114725_finelabels_DE_NK-like_CD8_T_cells_tumor_vs_normal.csv


In [3]:
# ----------------------------
# Cell 3 — Run pathway enrichment (Hallmark) per comparison, same
# viability logic as original notebook 06 (need >=5 significant genes
# for meaningful enrichment; background = genes actually tested).
# ----------------------------
summary_rows = []

for de_file in de_files:
    fname = Path(de_file).stem
    df = pd.read_csv(de_file, index_col=0)

    sig_genes = df[df["padj"] < 0.05].index.tolist()
    background_genes = df.index.tolist()

    if len(sig_genes) < 5:
        print(f"  SKIP {fname}: only {len(sig_genes)} significant genes (need >= 5)")
        continue

    try:
        enr = gp.enrich(
            gene_list=sig_genes,
            gene_sets="MSigDB_Hallmark_2020",
            background=background_genes,
            outdir=None,
        )
        results = enr.results
        results["Adjusted P-value"] = results["Adjusted P-value"].astype(float)
        n_sig_pathways = (results["Adjusted P-value"] < 0.05).sum()

        print(f"  {fname}: {len(sig_genes)} input genes (background={len(background_genes)}) "
              f"-> {n_sig_pathways} significant pathways (adj p<0.05)")

        results.to_csv(RESULTS_DIR / f"{fname}_pathways.csv", index=False)
        sig_results = results[results["Adjusted P-value"] < 0.05].sort_values("Adjusted P-value")
        sig_results.to_csv(RESULTS_DIR / f"{fname}_pathways_significant.csv", index=False)

        top_pathway = sig_results.iloc[0]["Term"] if len(sig_results) > 0 else \
            results.sort_values("Adjusted P-value").iloc[0]["Term"]
        top_pval = sig_results.iloc[0]["Adjusted P-value"] if len(sig_results) > 0 else \
            results.sort_values("Adjusted P-value").iloc[0]["Adjusted P-value"]

        summary_rows.append({
            "comparison": fname, "n_significant_pathways": n_sig_pathways,
            "top_pathway": top_pathway, "top_pathway_adj_pvalue": top_pval
        })
    except Exception as e:
        print(f"  FAILED {fname}: {type(e).__name__}: {e}")

summary_df = pd.DataFrame(summary_rows).sort_values("n_significant_pathways", ascending=False)
print(f"\n\nPathway enrichment complete: {len(summary_rows)} comparisons produced results")
summary_df.to_csv(RESULTS_DIR / "phase4_finelabels_pathway_summary_all.csv", index=False)
print(summary_df.to_string(index=False))

  SKIP GSE114725_finelabels_DE_Activated_CD8_T_cells_tumor_vs_normal: only 1 significant genes (need >= 5)
  SKIP GSE114725_finelabels_DE_B_cells_tumor_vs_normal: only 1 significant genes (need >= 5)
  SKIP GSE114725_finelabels_DE_CD4_Activated_T_cells_tumor_vs_normal: only 2 significant genes (need >= 5)
  GSE114725_finelabels_DE_Monocyte-like_macrophages_tumor_vs_normal: 37 input genes (background=3013) -> 0 significant pathways (adj p<0.05)
  SKIP GSE114725_finelabels_DE_NK-like_CD8_T_cells_tumor_vs_normal: only 0 significant genes (need >= 5)
  SKIP GSE114725_finelabels_DE_Resting_Resident_macrophages_tumor_vs_normal: only 2 significant genes (need >= 5)
  SKIP GSE114725_finelabels_DE_True_NK_cells_tumor_vs_normal: only 1 significant genes (need >= 5)
  GSE176078_finelabels_DE_B_cells_TNBC_vs_ER+: 79 input genes (background=5851) -> 10 significant pathways (adj p<0.05)
  GSE176078_finelabels_DE_CAFs_TNBC_vs_ER+: 11 input genes (background=13706) -> 1 significant pathways (adj p<0.0

In [4]:
# ----------------------------
# Compare Cycling epithelial's fine-label result against the original
# parent-level result, to check whether this is a genuine new finding
# or simply a replication (since Cycling epithelial was never
# sub-clustered — it's the same population at both resolutions).
# ----------------------------
PARENT_DE_DIR = PROJECT_DIR / "results" / "phase3_pseudobulk_de"
PARENT_PATHWAY_DIR = PROJECT_DIR / "results" / "phase3_pathway_enrichment"

# Compare DE gene counts
parent_de = pd.read_csv(PARENT_DE_DIR / "GSE176078_DE_Cycling_epithelial_TNBC_vs_ER+_significant.csv", index_col=0)
fine_de = pd.read_csv(DE_RESULTS_DIR / "GSE176078_finelabels_DE_Cycling_epithelial_TNBC_vs_ER+_significant.csv", index_col=0)

print(f"Parent-level significant genes: {len(parent_de)}")
print(f"Fine-label significant genes: {len(fine_de)}")

parent_genes = set(parent_de.index)
fine_genes = set(fine_de.index)
overlap = parent_genes & fine_genes
print(f"Overlap: {len(overlap)} genes")
print(f"Only in parent: {len(parent_genes - fine_genes)}")
print(f"Only in fine: {len(fine_genes - parent_genes)}")

# Compare pathway results
parent_pathways = pd.read_csv(PARENT_PATHWAY_DIR / "GSE176078_DE_Cycling_epithelial_TNBC_vs_ER+_pathways_significant.csv")
fine_pathways = pd.read_csv(RESULTS_DIR / "GSE176078_finelabels_DE_Cycling_epithelial_TNBC_vs_ER+_pathways_significant.csv")

print(f"\nParent-level significant pathways: {len(parent_pathways)}")
print(f"Fine-label significant pathways: {len(fine_pathways)}")
print(f"\nParent top pathway: {parent_pathways.iloc[0]['Term']}")
print(f"Fine top pathway: {fine_pathways.iloc[0]['Term']}")

Parent-level significant genes: 448
Fine-label significant genes: 448
Overlap: 448 genes
Only in parent: 0
Only in fine: 0

Parent-level significant pathways: 9
Fine-label significant pathways: 9

Parent top pathway: Estrogen Response Early
Fine top pathway: Estrogen Response Early


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
RES = PROJECT_DIR / "results" / "phase4_finelabels_pathways"
FIGURE_DIR = PROJECT_DIR / "figures" / "phase4_finelabels_pathways"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

def plot_pathway_dotplot(csv_path, title, save_path, top_n=15):
    df = pd.read_csv(csv_path)
    df = df.sort_values("Adjusted P-value").head(top_n).copy()
    df["neg_log10_adj_p"] = -np.log10(df["Adjusted P-value"].clip(lower=1e-300))
    df["n_genes"] = df["Genes"].apply(lambda g: len(str(g).split(";")))
    df = df.sort_values("neg_log10_adj_p")

    fig, ax = plt.subplots(figsize=(8, max(3, len(df) * 0.4)))
    sig_mask = df["Adjusted P-value"] < 0.05
    colors = np.where(sig_mask, "firebrick", "grey")
    ax.scatter(df["neg_log10_adj_p"], range(len(df)),
               s=df["n_genes"] * 30, c=colors, alpha=0.7, edgecolors="black", linewidth=0.5)
    ax.set_yticks(range(len(df)))
    ax.set_yticklabels(df["Term"], fontsize=9)
    ax.axvline(-np.log10(0.05), color="grey", linestyle="--", linewidth=0.8)
    ax.set_xlabel("-log10(adjusted p-value)")
    ax.set_title(title, fontsize=11)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"Saved: {save_path.name}")

# ----------------------------
# Cytotoxic CD8 T cells (reclassified) — both TNBC comparisons
# ----------------------------
plot_pathway_dotplot(
    RES / "GSE176078_finelabels_DE_Cytotoxic_CD8_T_cells_reclassified_TNBC_vs_ER+_pathways.csv",
    "GSE176078 — Cytotoxic CD8+ T cells, TNBC vs ER+",
    FIGURE_DIR / "GSE176078_CytotoxicCD8_pathways_TNBC_vs_ERplus.png"
)
plot_pathway_dotplot(
    RES / "GSE176078_finelabels_DE_Cytotoxic_CD8_T_cells_reclassified_TNBC_vs_HER2+_pathways.csv",
    "GSE176078 — Cytotoxic CD8+ T cells, TNBC vs HER2+",
    FIGURE_DIR / "GSE176078_CytotoxicCD8_pathways_TNBC_vs_HER2plus.png"
)

print("\nBoth Cytotoxic CD8 pathway figures generated.")

Saved: GSE176078_CytotoxicCD8_pathways_TNBC_vs_ERplus.png
Saved: GSE176078_CytotoxicCD8_pathways_TNBC_vs_HER2plus.png

Both Cytotoxic CD8 pathway figures generated.
